# Realistic SpAM Simulation & MDS Evaluation Pipeline

This notebook mirrors `evaluation.ipynb`, but simulates subjects under the **same per-subject
trial design as the deployed task** (`SpAM_Task/`) instead of drawing every trial uniformly
from the whole image pool. Concretely, each subject:

1. Is restricted to their own random `n_unique`-image subset of the full pool (derived from
   `trials_per_subject`, `images_per_trial`, `frac_images_repeated` - see
   `SpAM_Simulations/design.py::compute_design_counts`).
2. Has that subset allocated into trials via the same 3-pass greedy algorithm as
   `SpAM_Task/js/trial_generator.js`'s `buildTrialLists`, with `frac_images_repeated` of the
   subset shown in exactly 2 distinct trials (the rest in exactly 1).
3. Additionally yields a per-subject **SNR heuristic**:
   $$\mathrm{SNR} = \frac{\sigma_d}{\mathrm{mean}(|\Delta d|)}$$
   where $\sigma_d$ is the within-subject std of that subject's own observed (noisy)
   pairwise distances, and $\Delta d$ is the difference between the two independent noisy
   measurements of the same image pair, for pairs the subject happened to observe twice
   (only possible when both images are "doubly-presented" and land on the same two trials -
   see `realistic_experiment._find_candidate_repeated_pairs`). **This is NaN for subjects
   with no such repeated pairs** - a real property of the heuristic, not a bug; small
   `(trials_per_subject, images_per_trial, frac_images_repeated)` configurations can yield
   zero repeated pairs for some subjects by chance.

Everything downstream of trial generation - ground-truth generation, the MDS sweep
(`multi_dimensional_scaling.py`/R `smacof`), coverage/stability metrics, and on-disk storage
- is reused unchanged from the original pipeline (`SpAM_Simulations/pipeline.py`).


In [ ]:
from pathlib import Path
from itertools import combinations, product

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express.colors as px_colors
import plotly.io as pio

from SpAM_Simulations.config import RealisticSimulationConfig, MDSSweepConfig
from SpAM_Simulations import pipeline

pio.renderers.default = 'browser'


## Configure & Generate Simulation
Pick a `RealisticSimulationConfig` - same levers as `SimulationConfig`
(`n_images`/`n_dims`, `num_subjects`, `trials_per_subject`, `images_per_trial`,
`subjects_noise_scale`, `subjects_noise_df`, `reps`, `seed`) plus the new
`frac_images_repeated`. The bundled small config runs end-to-end quickly; uncomment the
full-study config (matching `SpAM_Task/task_config.json`'s real `t=10, k=20` and the
725-image dataset) for the real run.


In [ ]:
# --- Study configuration ----------------------------------------------------------------
# Small configuration for a quick end-to-end run:
sim_config = RealisticSimulationConfig(
    n_images=300,
    n_dims=6,
    num_subjects=[20, 40, 75],
    trials_per_subject=[10],
    images_per_trial=[20],
    subjects_noise_scale=[0.0, 0.3, 0.6],
    subjects_noise_df=[3],
    frac_images_repeated=[0.0, 0.2, 1 / 3],
    reps=3,
    seed=42,
)

# Full study configuration (uncomment for the real run - this is much heavier, and mirrors
# the deployed task's t=10/k=20 design over the full 725-image dataset):
# sim_config = RealisticSimulationConfig(
#     n_images=725,
#     n_dims=10,
#     num_subjects=[20, 30, 50, 75, 250],
#     trials_per_subject=[10],
#     images_per_trial=[20],
#     subjects_noise_scale=[0.0, 0.2, 0.5, 0.8],
#     subjects_noise_df=[1, 5],
#     frac_images_repeated=[0.0, 0.2, 1 / 3, 0.45],
#     reps=5,
#     seed=42,
# )

sim = pipeline.generate_realistic_simulation(sim_config, verbose=True)


## Evaluate Simulation
### Coverage
Same three coverage scores as the original notebook (image coverage, pair coverage,
`P[connected]`), now also broken out by `frac_images_repeated`: a higher fraction of
repeated images means fewer *distinct* images per subject (`n_unique` shrinks), which can
lower coverage at a fixed subject count.


In [ ]:
coverage = pipeline.compute_coverage_table(sim).rename(columns={"num_subjects": "n_subjects"})
coverage["is_connected"] = coverage["num_connected_components"] == 1.0

# Coverage scores aren't affected by subject noise, so we average across those parameters
# (and across trials_per_subject/images_per_trial here, since the bundled config fixes them -
# extend the groupby below if you sweep those too).
coverage_summary = (
    coverage.groupby(["n_subjects", "frac_images_repeated"])
    .agg(
        num_reps=("img_coverage", "size"),
        percent_img_coverage_mean=("img_coverage", "mean"),
        percent_img_coverage_sem=("img_coverage", "sem"),
        percent_pair_coverage_mean=("pair_coverage", "mean"),
        percent_pair_coverage_sem=("pair_coverage", "sem"),
        p_is_connected_mean=("is_connected", "mean"),
        p_is_connected_sem=("is_connected", "sem"),
    )
    .sort_index()
    .reset_index()
)


In [ ]:
ROW_TITLES = {
    "% IMG COVERAGE": "percent_img_coverage",
    "% PAIR COVERAGE": "percent_pair_coverage",
    "P[CONNECTED]": "p_is_connected",
}
frac_values = sorted(coverage_summary["frac_images_repeated"].unique())
COL_TITLES = {f: f"frac_images_repeated = {f:.3g}" for f in frac_values}
coverage_fig = make_subplots(
    rows=len(ROW_TITLES), cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="Number of Subjects",
    vertical_spacing=0.05, horizontal_spacing=0.025,
)
for c, frac in enumerate(frac_values):
    df = coverage_summary[coverage_summary["frac_images_repeated"] == frac]
    for r, (row_title, prefix) in enumerate(ROW_TITLES.items()):
        coverage_fig.add_trace(
            row=r + 1, col=c + 1, trace=go.Scatter(
                x=df["n_subjects"], y=df[f"{prefix}_mean"],
                error_y=dict(type="data", array=df[f"{prefix}_sem"], visible=True),
                mode="lines+markers", line=dict(color=px_colors.qualitative.Plotly[r]),
                showlegend=False,
            )
        )
        if c == 0:
            coverage_fig.update_yaxes(
                row=r + 1, col=c + 1,
                title=dict(text=row_title, font=dict(size=14, color='black'))
            )
del c, r, frac, df, row_title, prefix

coverage_fig.update_layout(
    height=650, width=1500,
    title=dict(
        text="Coverage by Number of Subjects and Image-Repetition Fraction",
        font=dict(size=20, color='black')
    ),
)
coverage_fig.show()


### Stability
Spearman (rank) correlation between different repetitions of the same experimental
configuration, faceted by `frac_images_repeated` and colored by the noise parameters - same
idea as the original notebook.


In [ ]:
PARAM_FIELDS = [
    "num_subjects", "trials_per_subject", "images_per_trial",
    "subjects_noise_scale", "subjects_noise_df", "frac_images_repeated",
]

correlations = (
    pipeline.compute_stability_table(sim)
    .dropna(subset=["spearman"])
    .groupby(PARAM_FIELDS)
    .agg(count=("spearman", "size"), r_mean=("spearman", "mean"), r_sem=("spearman", "sem"))
)
correlations.index = correlations.index.set_names(
    ["n_subjects", "trials_per_subject", "images_per_trial",
     "subjects_noise_scale", "subjects_noise_df", "frac_images_repeated"]
)


In [ ]:
frac_values = sorted(correlations.index.get_level_values("frac_images_repeated").unique())
COL_TITLES = {f: f"frac_images_repeated = {f:.3g}" for f in frac_values}
corr_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="Number of Subjects",
)
for c, frac in enumerate(frac_values):
    subset = correlations.loc[correlations.index.get_level_values("frac_images_repeated") == frac]
    noise_scales = sorted(subset.index.get_level_values("subjects_noise_scale").unique())
    noise_dfs = sorted(subset.index.get_level_values("subjects_noise_df").unique())
    for i, (noise_scale, noise_df) in enumerate(product(noise_scales, noise_dfs)):
        name = f"Scale={noise_scale}<br>DFs={noise_df}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = subset.loc[
            (subset.index.get_level_values("subjects_noise_scale") == noise_scale) &
            (subset.index.get_level_values("subjects_noise_df") == noise_df)
        ]
        corr_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df.index.get_level_values("n_subjects"), y=df["r_mean"],
                error_y=dict(type="data", array=df["r_sem"], visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
            )
        )
del c, frac, subset, noise_scales, noise_dfs, i, noise_scale, noise_df, name, color, df

corr_fig.update_yaxes(row=1, col=1, title=dict(text="Spearman R", font=dict(size=14, color='black')))
corr_fig.update_layout(
    height=450, width=1500,
    title=dict(text="Pre-MDS Stability by Image-Repetition Fraction", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subject Noise Parameters")),
)
corr_fig.show()


## Subject SNR Heuristic
Each subject also yields an `SNR = sigma_d / mean(|delta_d|)` heuristic (see the intro
cell), computed only from pairs the subject happened to observe twice via the
`frac_images_repeated` mechanic - exactly the kind of signal we could also compute from real
(ground-truth-free) data. Two checks below: (1) its distribution across subjects, and (2)
whether it actually tracks the configured noise lever - if it didn't, it would be useless as
a real-data QC proxy.


In [ ]:
# Per-subject SNR for one representative configuration: most repeated images (so repeated
# pairs - and thus a defined SNR - actually exist), most subjects, heaviest-tailed and
# noisiest, across its repetitions.
focus_params = max(
    sim._results,
    key=lambda p: (p.frac_images_repeated, p.num_subjects, -p.subjects_noise_df, p.subjects_noise_scale),
)
all_snr = np.concatenate([res.subject_snr for res in sim._results[focus_params]])
finite_snr = all_snr[np.isfinite(all_snr)]

snr_hist_fig = go.Figure(go.Histogram(x=finite_snr, nbinsx=30))
snr_hist_fig.update_layout(
    width=700, height=400,
    title=dict(text=f"Subject SNR Distribution<br><sup>{focus_params}</sup>"),
    xaxis=dict(title=dict(text="SNR = sigma_d / mean(|delta_d|)")),
    yaxis=dict(title=dict(text="Count")),
    template="plotly_white",
)
snr_hist_fig.show()
print(f"{np.mean(np.isnan(all_snr)):.1%} of subjects had no repeated pairs (SNR undefined)")


In [ ]:
# Excludes subjects_noise_scale == 0.0: noiseless subjects have SNR = inf (not finite), and
# the heuristic is only meaningful once there's noise to estimate.
snr_vs_noise = (
    coverage[np.isfinite(coverage["mean_snr"])]
    .groupby(["subjects_noise_scale", "frac_images_repeated"])
    .agg(mean_snr=("mean_snr", "mean"), sem_snr=("mean_snr", "sem"))
    .reset_index()
)

snr_fig = go.Figure()
for frac in sorted(snr_vs_noise["frac_images_repeated"].unique()):
    df = snr_vs_noise[snr_vs_noise["frac_images_repeated"] == frac]
    snr_fig.add_trace(go.Scatter(
        x=df["subjects_noise_scale"], y=df["mean_snr"],
        error_y=dict(type="data", array=df["sem_snr"].fillna(0), visible=True),
        name=f"frac_images_repeated = {frac:.3g}",
        mode="lines+markers",
    ))
del frac, df
snr_fig.update_layout(
    width=700, height=400,
    title=dict(text="Mean SNR Heuristic vs. Configured Subject Noise Scale"),
    xaxis=dict(title=dict(text="subjects_noise_scale")),
    yaxis=dict(title=dict(text="mean(SNR)"), type="log"),
    template="plotly_white",
    legend=dict(title=dict(text="Image-Repetition Fraction")),
)
snr_fig.show()


## Run MDS
Same MDS sweep as the original notebook - it operates only on each configuration's mean
observed distances + weight mask, so it's entirely agnostic to how those distances were
generated (uniform-random trials vs. the realistic per-subject design here).


In [ ]:
MDS_STORE_PATH = Path("mds_store") / "realistic"
sweep_config = MDSSweepConfig(
    min_ndim=2,
    max_iters=500,
    convergence_tol=1e-6,
    precalc_init=False,
)
store = pipeline.run_mds_sweep(sim, sweep_config, MDS_STORE_PATH, parallel=False, verbose=True)
mds_meta = store.metadata()
print(f"{len(mds_meta)} MDS results stored at {MDS_STORE_PATH}")


In [ ]:
from collections import Counter

Counter(mds_meta["status"])


### Scree Plot
MDS Stress vs target dimensionality, faceted by `frac_images_repeated`.


In [ ]:
success_mds_results = mds_meta[mds_meta["status"].isin(["success", "max_iters"])].copy()

frac_values = sorted(success_mds_results["frac_images_repeated"].unique())
COL_TITLES = {f: f"frac_images_repeated = {f:.3g}" for f in frac_values}
stress_scree_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="DIMENSIONS",
)
for c, frac in enumerate(frac_values):
    subset = success_mds_results[success_mds_results["frac_images_repeated"] == frac]
    noise_scales = sorted(subset["subjects_noise_scale"].unique())
    n_subjs = sorted(subset["num_subjects"].unique())
    for i, (noise_scale, n_subj) in enumerate(product(noise_scales, n_subjs)):
        name = f"Scale={noise_scale}<br>N={n_subj}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = (
            subset[(subset["subjects_noise_scale"] == noise_scale) & (subset["num_subjects"] == n_subj)]
            .groupby("ndim")["stress"]
            .agg(N="count", mean="mean", sem="sem")
        )
        stress_scree_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df.index.get_level_values("ndim"), y=df["mean"],
                error_y=dict(type="data", array=df["sem"].fillna(0), visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
                hovertemplate=f"{name}<br>" + "NDIM=%{x}<br>Stress=%{y:.4f}",
            )
        )
del c, frac, subset, noise_scales, n_subjs, i, noise_scale, n_subj, name, color, df

stress_scree_fig.update_yaxes(row=1, col=1, title=dict(text="Stress", font=dict(size=14, color='black')))
stress_scree_fig.update_layout(
    height=500, width=1500,
    title=dict(text="MDS Stress (lower is better)", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subjects / Noise")),
)
stress_scree_fig.show()


### Embedding Stability
Post-MDS Spearman agreement of reconstructed distances (`confdist`) across repetitions,
faceted by `frac_images_repeated`.


In [ ]:
embedding_corrs_df = (
    pipeline.compute_embedding_stability(store)
    .rename(columns={"n_reps": "N", "mean_spearman": "r_mean", "sem_spearman": "r_sem"})
)

frac_values = sorted(embedding_corrs_df["frac_images_repeated"].unique())
COL_TITLES = {f: f"frac_images_repeated = {f:.3g}" for f in frac_values}
embedded_corr_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="DIMENSIONS",
)
for c, frac in enumerate(frac_values):
    subset = embedding_corrs_df[embedding_corrs_df["frac_images_repeated"] == frac]
    noise_scales = sorted(subset["subjects_noise_scale"].unique())
    n_subjs = sorted(subset["num_subjects"].unique())
    for i, (noise_scale, n_subj) in enumerate(product(noise_scales, n_subjs)):
        name = f"Scale={noise_scale}<br>N={n_subj}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = subset[(subset["subjects_noise_scale"] == noise_scale) & (subset["num_subjects"] == n_subj)]
        embedded_corr_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df["ndim"], y=df["r_mean"],
                error_y=dict(type="data", array=df["r_sem"].fillna(0), visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
                hovertemplate=f"{name}<br>" + "NDIM=%{x}<br>Spearman's R=%{y:.4f}",
            )
        )
del c, frac, subset, noise_scales, n_subjs, i, noise_scale, n_subj, name, color, df

embedded_corr_fig.update_yaxes(row=1, col=1, title=dict(text="Spearman R", font=dict(size=14, color='black')))
embedded_corr_fig.update_layout(
    height=500, width=1500,
    title=dict(text="MDS-Stability (Spearman Correlation) by Image-Repetition Fraction", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subjects / Noise")),
)
embedded_corr_fig.show()
